In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

os.chdir(PROJECT_ROOT)

print("Working directory:", Path.cwd())
print("Source directory:", PROJECT_ROOT / "src")

Working directory: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026
Source directory: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026/src


In [3]:
from dotenv import load_dotenv
import os

load_dotenv(PROJECT_ROOT / ".env")

print("PG_CONN_STR configured:", bool(os.getenv("PG_CONN_STR")))

PG_CONN_STR configured: True


In [ ]:
from index import text_search, vector_search, hybrid_search

print("Retrieval functions imported successfully.")

Retrieval functions imported successfully.


In [5]:
import psycopg

with psycopg.connect(os.environ["PG_CONN_STR"]) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                doc_id,
                symbol,
                year,
                quarter,
                COUNT(*) AS chunk_count
            FROM earnings_chunks
            GROUP BY doc_id, symbol, year, quarter
            ORDER BY symbol, year DESC, quarter DESC
        """)

        transcript_rows = cur.fetchall()

for row in transcript_rows:
    print(row)

('AAPL_2026Q3', 'AAPL', 2026, 3, 50)
('AMD_2026Q2', 'AMD', 2026, 2, 51)
('AVGO_2026Q2', 'AVGO', 2026, 2, 39)
('GOOG_2026Q2', 'GOOG', 2026, 2, 54)
('LRCX_2026Q4', 'LRCX', 2026, 4, 69)
('META_2026Q2', 'META', 2026, 2, 61)
('MSFT_2026Q4', 'MSFT', 2026, 4, 58)
('NVDA_2027Q1', 'NVDA', 2027, 1, 47)
('TSLA_2026Q2', 'TSLA', 2026, 2, 51)
('TSM_2026Q2', 'TSM', 2026, 2, 50)


In [7]:
# test retrieval manually
query = "What did Google say about AI"

for method_name, search_function in {
    "text": text_search,
    "vector": vector_search,
    "hybrid": hybrid_search,
}.items():
    print(f"\n=== {method_name.upper()} ===")

    results = search_function(query, limit=5)

    for rank, result in enumerate(results, start=1):
        print(
            rank,
            result["doc_id"],
            result["symbol"],
            result["year"],
            result["quarter"],
            result.get("score", result.get("rrf_score")),
        )


=== TEXT ===
Normalized query for text search: google | ai
1 GOOG_2026Q2 GOOG 2026 2 0.46290323
2 GOOG_2026Q2 GOOG 2026 2 0.44242424
3 GOOG_2026Q2 GOOG 2026 2 0.27857143
4 GOOG_2026Q2 GOOG 2026 2 0.07272727
5 GOOG_2026Q2 GOOG 2026 2 0.06410257

=== VECTOR ===
1 META_2026Q2 META 2026 2 0.5441794071090584
2 META_2026Q2 META 2026 2 0.5067120488457659
3 META_2026Q2 META 2026 2 0.47037187395263547
4 META_2026Q2 META 2026 2 0.46928298839533467
5 INTC_2026Q2 INTC 2026 2 0.45180922746658647

=== HYBRID ===
Normalized query for text search: google | ai
1 GOOG_2026Q2 GOOG 2026 2 0.46290323
2 META_2026Q2 META 2026 2 0.5441794071090584
3 GOOG_2026Q2 GOOG 2026 2 0.44242424
4 META_2026Q2 META 2026 2 0.5067120488457659
5 GOOG_2026Q2 GOOG 2026 2 0.27857143


In [3]:
import sys
print(sys.version)
print(sys.executable)

3.11.5 (main, Sep 11 2023, 08:19:27) [Clang 14.0.6 ]
/Users/camillecu/Downloads/KUL/llm_project/llm_project2026/llmproject311/bin/python


# export the dataframe

In [6]:
import os
import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv()

with psycopg.connect(os.environ["PG_CONN_STR"]) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT id, doc_id, chunk_index, symbol, year, quarter, date, source, text
            FROM earnings_chunks
            ORDER BY symbol, year, quarter, chunk_index
        """)
        rows = cur.fetchall()
        cols = [desc[0] for desc in cur.description]

df = pd.DataFrame(rows, columns=cols)

print("DataFrame shape:", df.shape)


DataFrame shape: (530, 9)


In [8]:
# export df to CSV
output_csv_path = PROJECT_ROOT/ "df.csv"
df.to_csv(output_csv_path, index=False)
print(f"DataFrame exported to CSV at: {output_csv_path}")

DataFrame exported to CSV at: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026/df.csv


# full comparison

In [17]:
from evaluate_retrieval import (
    evaluate_search_function,
    load_ground_truth,
    print_failures,
)

ground_truth = load_ground_truth(
    "data/retrieval_ground_truth.csv"
)

ground_truth_retrieval = [
    record
    for record in ground_truth
    if "NEEDS_CONTEXT" not in record["expected_chunk_ids"]
]

print("Retrieval-evaluation rows:", len(ground_truth_retrieval))

Retrieval-evaluation rows: 30


In [18]:
print(ground_truth_retrieval)

[{'query': 'What did Apple say about the upcoming advanced manufacturing center in Houston?', 'expected_doc_ids': {'AAPL_2026Q3'}, 'expected_chunk_ids': {'AAPL_2026Q3:13'}}, {'query': 'What is the Apple Upgrade program and what benefits does it offer to customers?', 'expected_doc_ids': {'AAPL_2026Q3'}, 'expected_chunk_ids': {'AAPL_2026Q3:30'}}, {'query': "How are changes to the App Store business model and the Supreme Court appeal impacting Apple's services?", 'expected_doc_ids': {'AAPL_2026Q3'}, 'expected_chunk_ids': {'AAPL_2026Q3:39'}}, {'query': 'How is AMD managing its supply chain for CPU, GPU, and networking components?', 'expected_doc_ids': {'AMD_2026Q2'}, 'expected_chunk_ids': {'AMD_2026Q2:43'}}, {'query': 'What did Lisa Su say about the trajectory and guidance for the server CPU market in 2027?', 'expected_doc_ids': {'AMD_2026Q2'}, 'expected_chunk_ids': {'AMD_2026Q2:40'}}, {'query': 'How did AMD address the question regarding supply constraints in the server CPU business?', 'e

In [ ]:
from index import text_search, vector_search, hybrid_search

search_methods = {
    "text": text_search,
    "vector": vector_search,
    "hybrid": hybrid_search,
}

evaluations = {}

for method_name, search_function in search_methods.items():
    evaluation = evaluate_search_function(
        search_function=search_function,
        ground_truth=ground_truth_retrieval,
        k=5,
    )

    evaluations[method_name] = evaluation

    print(f"\n=== {method_name.upper()} ===")
    print(f"Document Hit Rate@5: {evaluation['doc_hit_rate']:.3f}")
    print(f"Exact Chunk Hit Rate@5: {evaluation['chunk_hit_rate']:.3f}")
    print(f"Document MRR@5: {evaluation['doc_mrr']:.3f}")
    print(f"Exact Chunk MRR@5: {evaluation['chunk_mrr']:.3f}")


=== TEXT ===
Document Hit Rate@5: 0.833
Exact Chunk Hit Rate@5: 0.567
Document MRR@5: 0.789
Exact Chunk MRR@5: 0.483

=== VECTOR ===
Document Hit Rate@5: 0.700
Exact Chunk Hit Rate@5: 0.300
Document MRR@5: 0.667
Exact Chunk MRR@5: 0.267

=== HYBRID ===
Document Hit Rate@5: 0.833
Exact Chunk Hit Rate@5: 0.567
Document MRR@5: 0.740
Exact Chunk MRR@5: 0.418


# after changing chunking size

In [19]:
from index import text_search, vector_search, hybrid_search

search_methods = {
    "text": text_search,
    "vector": vector_search,
    "hybrid": hybrid_search,
}

evaluations = {}

for method_name, search_function in search_methods.items():
    evaluation = evaluate_search_function(
        search_function=search_function,
        ground_truth=ground_truth_retrieval,
        k=5,
    )

    evaluations[method_name] = evaluation

    print(f"\n=== {method_name.upper()} ===")
    print(f"Document Hit Rate@5: {evaluation['doc_hit_rate']:.3f}")
    print(f"Exact Chunk Hit Rate@5: {evaluation['chunk_hit_rate']:.3f}")
    print(f"Document MRR@5: {evaluation['doc_mrr']:.3f}")
    print(f"Exact Chunk MRR@5: {evaluation['chunk_mrr']:.3f}")


=== TEXT ===
Document Hit Rate@5: 1.000
Exact Chunk Hit Rate@5: 0.967
Document MRR@5: 0.967
Exact Chunk MRR@5: 0.815

=== VECTOR ===
Document Hit Rate@5: 0.600
Exact Chunk Hit Rate@5: 0.167
Document MRR@5: 0.544
Exact Chunk MRR@5: 0.167

=== HYBRID ===
Document Hit Rate@5: 1.000
Exact Chunk Hit Rate@5: 0.933
Document MRR@5: 0.961
Exact Chunk MRR@5: 0.736


# with rewriting

In [3]:
from evaluate_retrieval import (
    evaluate_search_function,
    load_ground_truth,
    print_failures,
)

ground_truth = load_ground_truth(
    "data/retrieval_ground_truth.csv"
)

ground_truth_retrieval = [
    record
    for record in ground_truth
    if "NEEDS_CONTEXT" not in record["expected_chunk_ids"]
]

print("Retrieval-evaluation rows:", len(ground_truth_retrieval))

Retrieval-evaluation rows: 30


In [ ]:
from index_rewrite import text_search, vector_search, hybrid_search

search_methods = {
    "text": text_search,
    "vector": vector_search,
    "hybrid": hybrid_search,
}

evaluations = {}

for method_name, search_function in search_methods.items():
    evaluation = evaluate_search_function(
        search_function=search_function,
        ground_truth=ground_truth_retrieval,
        k=5,
    )

    evaluations[method_name] = evaluation

    print(f"\n=== {method_name.upper()} ===")
    print(f"Document Hit Rate@5: {evaluation['doc_hit_rate']:.3f}")
    print(f"Exact Chunk Hit Rate@5: {evaluation['chunk_hit_rate']:.3f}")
    print(f"Document MRR@5: {evaluation['doc_mrr']:.3f}")
    print(f"Exact Chunk MRR@5: {evaluation['chunk_mrr']:.3f}")

Normalized query for text search: apple | report | for | total | revenue | in | fiscal | q3 | 2026
Normalized query for text search: gross | margin | apple | report | drove | sequential | change
Normalized query for text search: apple | guide | for | revenue | growth | gross | margin | in | september | quarter
Normalized query for text search: supply | constraints | apple | expect | affect | iphone | mac | ipad
Normalized query for text search: apple | rising | memory | costs | their | impact | beyond | september
Normalized query for text search: alphabet's | revenue | growth | in | q2 | 2026
Normalized query for text search: how | google | cloud | revenue | operating | margin | perform
Normalized query for text search: alphabet | guide | for | full-year | 2026 | capex | why | it | increased
Normalized query for text search: google | ai | infrastructure | demand | supply | constraints
Normalized query for text search: how | alphabet | describe | ai | opportunity | expected | roic
Norma

# with rewriting after changing chunking size

In [20]:
from index_rewrite import text_search, vector_search, hybrid_search

search_methods = {
    "text": text_search,
    "vector": vector_search,
    "hybrid": hybrid_search,
}

evaluations = {}

for method_name, search_function in search_methods.items():
    evaluation = evaluate_search_function(
        search_function=search_function,
        ground_truth=ground_truth_retrieval,
        k=5,
    )

    evaluations[method_name] = evaluation

    print(f"\n=== {method_name.upper()} ===")
    print(f"Document Hit Rate@5: {evaluation['doc_hit_rate']:.3f}")
    print(f"Exact Chunk Hit Rate@5: {evaluation['chunk_hit_rate']:.3f}")
    print(f"Document MRR@5: {evaluation['doc_mrr']:.3f}")
    print(f"Exact Chunk MRR@5: {evaluation['chunk_mrr']:.3f}")

Normalized query for text search: apple or upcoming or advanced or manufacturing or center or in or houston
Normalized query for text search: apple or upgrade or program or benefits or it or offer or customers
Normalized query for text search: how or changes or app or store or business or model or supreme or court or appeal or impacting or apple's or services
Normalized query for text search: how or amd or managing or its or supply or chain or for or cpu or gpu or networking or components
Normalized query for text search: lisa or su or trajectory or guidance or for or server or cpu or market or in or 2027
Normalized query for text search: how or amd or address or question or regarding or supply or constraints or in or server or cpu or business
Normalized query for text search: how or xpu or platform or help or provide or compute or capacity or for or frontier or model or customers
Normalized query for text search: how or far or out or broadcom's or visibility or run or due or huge or d